#### Core Idea: Predict Scoring

- Input: mix = `List[(sound_id, volume)]`
- Output: Top K Ranked Candidate = `List[(sound_id, predicted_volume)]`

Steps:
1. Candidate Generation
2. Scoring + Ranking


**1. Candidate Generation**

K = 50

- candidates = Union[topkcooccurences(mix), topk-content-similarity(mix)] - set(mix_sounds_id)
    * co-occurence: content that appear often together
    * similar content: content similar to each other (using tags)

**2. Scoring + Ranking**

```
score = (
    α * normalize(content_sim)
  + β * normalize(co_occ)
  + γ * normalize(pop_penalty)
  + δ * normalize(-diversity_penalty)
)

α = 0.3   # content
β = 0.5   # co-occurrence (strongest)
γ = 0.1   # popularity
δ = 0.1   # diversity

```

**How**

Content Similarity (Mix similarity)
- Each sound become a vector => rain = [1, 0, 1, ..] where each feature correspond to a tag
- Each mix is a combination of sounds ie average of sounds embedding weighted by volume
- We can compute mix/sound similarity using distance (co-sine, ...)

Co-occurence Matrix
- Count each pair together => (rain, thunder) = 2; (rain, wind) = 5, ...
- Create co-occurence matrix `[n_sounds x n_sounds]`
- To get mix co-occurence, average co-occurence => ex: mix = [rain, wind], then `co-occ(mix) = (co-occ(rain) + co-occ(wind))/ 2`

Popularity Penalty
- How often does a sound appear in a mix => `pop(sound) = sum(sounds_exploded) / total_mixes` => sound_id appear X% of the time
- `pop_penalty(s) =  (log(1 + count(s)) - mean) / std`

Diversity
- Don't recommend something too similar to what's already there to not be redundant => ex: if mix = [rain, rain_light], don't recommend rain_soft_2 (boring); thunder is more interesting
- Compute redundancy => `redundancy(mix, sound_j) = max_{sound_i in mix} similarity(sound_i, sound_j)`
- Diversity within recommendation list: `MMR(c)=λ⋅relevance(c)−(1−λ)⋅r∈selectedmax​sim(c,r)`, where relevance is main
- 




In [ ]:
import json
import pandas as pd

In [ ]:
SOUNDS_TAG_PATH = "~/Data/ContentAirTable/content_database_sounds.csv"
MUSIC_TAG_PATH = "~/Data/ContentAirTable/content_database_music.csv"
MIXES_TAG_PATH = "~/Data/ContentAirTable/content_database_mixes.csv"
SLEEPTALES_TAG_PATH = "~/Data/ContentAirTable/content_database_sleeptales.csv"
MEDITATIONS_TAG_PATH = "~/Data/ContentAirTable/content_database_meditations.csv"

SOUNDS_IDS_JSON_PATH = "../data/sounds_ids.json"
SOUNDS_IDS_TO_CONTENT_IDS_JSON_PATH = "../data/sounds_id_to_content_id.json"
LISTENING_CSV = "/Users/emulie/Downloads/script_job_67b7f56b753852fd2a3f35baed75edbc_0.csv"

In [ ]:
df_sounds = pd.read_csv(SOUNDS_TAG_PATH)
df_music = pd.read_csv(MUSIC_TAG_PATH)
df_mixes = pd.read_csv(MIXES_TAG_PATH)
df_sleeptales = pd.read_csv(SLEEPTALES_TAG_PATH)
df_meditations = pd.read_csv(MEDITATIONS_TAG_PATH)
df_listening = pd.read_csv(LISTENING_CSV)

df_listening['sounds'] = df_listening['sounds'].apply(json.loads)
df_listening['sounds_volume'] = df_listening['sounds_volume'].apply(json.loads)

In [ ]:
with open(SOUNDS_IDS_JSON_PATH) as f:
    sounds_ids = json.load(f)

with open(SOUNDS_IDS_TO_CONTENT_IDS_JSON_PATH) as f:
    sound_to_content_ids = json.load(f)

In [ ]:
# sounds_ids
sound_to_content_ids

In [ ]:
# list(df_sounds['Content ID'].unique())

In [ ]:
sounds_music_columns = ['Content ID', ]
df_sounds

In [ ]:
df_sounds.columns

In [ ]:
print(len(sounds_ids))
print(len(sound_to_content_ids['mapped']))

In [ ]:
print(set(sounds_ids) - set(sound_to_content_ids['mapped'].keys()))

### Co-occ Matrix

Biases:
1. Popularity bias (biggest issue)
ambience.rain appears in 60% of sessions, so it co-occurs with everything by chance. Its counts are inflated regardless of actual affinity.

2. Session length bias
A 5-sound session generates 10 pairs, a 2-sound session generates 1. Sounds that appear in longer mixes get disproportionately high counts.

3. Repetition bias
Users who found a good combo keep reusing it. You're counting the same user's habit many times, not independent evidence of a good pair.

4. Recency/exposure bias
Older or more prominently featured sounds had more chances to accumulate co-occurrences — not because they're better matches.

Alternative:
- Use ALS


In [ ]:
df_listening_exploded = df_listening.explode(['sounds', 'sounds_volume'])
df_listening_exploded

In [ ]:
from collections import defaultdict
from scipy.sparse import csr_matrix

# Each row in df_listening is a session with a list of sounds
# Co-occurrence: how many sessions contain both sound_i and sound_j

sound_labels = sorted(df_listening_exploded['sounds'].unique())
sound_to_idx = {s: i for i, s in enumerate(sound_labels)}
n = len(sound_labels)

co_counts = defaultdict(int)
for sounds in df_listening['sounds']:
    unique_sounds = list(set(sounds))          # deduplicate within session
    for i in range(len(unique_sounds)):
        for j in range(i + 1, len(unique_sounds)):
            pair = tuple(sorted([unique_sounds[i], unique_sounds[j]]))
            co_counts[pair] += 1

# Build symmetric sparse matrix
rows, cols, data = [], [], []
for (s1, s2), c in co_counts.items():
    i, j = sound_to_idx.get(s1), sound_to_idx.get(s2)
    if i is not None and j is not None:
        rows += [i, j]
        cols += [j, i]
        data += [c, c]

co_occ_matrix = csr_matrix((data, (rows, cols)), shape=(n, n))

# Sanity check
print(f"Matrix shape: {co_occ_matrix.shape}")
print(f"Non-zero pairs: {co_occ_matrix.nnz // 2:,}")

# Most frequent co-occurring pairs
top_pairs = sorted(co_counts.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_pairs, columns=['pair', 'co_count'])


In [ ]:
import numpy as np
co_occ_dense = co_occ_matrix.toarray()

def co_occ_score(mix_sounds, candidate_id):
    """Mean co-occurrence count between candidate and each sound in the mix."""
    j = sound_to_idx.get(candidate_id)
    if j is None:
        return 0.0
    scores = [
        co_occ_dense[sound_to_idx[s], j]
        for s in mix_sounds
        if s in sound_to_idx
    ]
    return np.mean(scores) if scores else 0.0


In [ ]:
# df_listening['sounds']
df_listening['sounds'].iloc[71304]

### JOIN

In [ ]:
TAG_COLS_EDA = ['ℹ️ Themes', 'ℹ️ User Needs', 'ℹ️ Environment', 'ℹ️ Sound Features']
TAG_COL_LABELS = {
    'Content ID': 'content_id',
    'ℹ️ Themes':        'themes',
    'ℹ️ User Needs':    'user_needs',
    'ℹ️ Environment':   'environment',
    'ℹ️ Sound Features':'sound_features',
    # 'ℹ️ Linked Content(s)': 'linked_content',
}

df_sounds_cleaned = df_sounds[TAG_COL_LABELS.keys()].rename(columns=TAG_COL_LABELS)
df_musics_cleaned = df_music[TAG_COL_LABELS.keys()].rename(columns=TAG_COL_LABELS)
df_sounds_music = pd.concat([df_sounds_cleaned, df_musics_cleaned])

In [ ]:
df_sounds_music

In [ ]:
# get all themes, user_needs, environment, sound_features + Counts
all_themes = df_sounds_music['themes'].str.split(',').explode(['themes']).unique().tolist()
all_user_needs = df_sounds_music['user_needs'].str.split(',').explode(['user_needs']).unique().tolist()
all_environments = df_sounds_music['environment'].str.split(',').explode(['environment']).unique().tolist()
all_sounds_feature = df_sounds_music['sound_features'].str.split(',').explode(['sound_features']).unique().tolist()

In [ ]:
print(f"{all_themes=}")
print(f"{all_user_needs=}")
print(f"{all_environments=}")
print(f"{all_sounds_feature=}")
print(f"{len(df_sounds_music)=}")

In [ ]:
# check for popular tags: filter out those in 80%
all_tags = all_themes + all_user_needs + all_environments + all_sounds_feature
# df_tags = 

def build_tag_matrix(df):
    # Parse comma-separated strings into lists
    tag_cols = ['themes', 'user_needs', 'environment', 'sound_features']
    
    def parse_tags(val):
        if pd.isna(val) or val == 'NaN' or val == '':
            return []
        return [t.strip() for t in str(val).split(',')]
    
    # Collect all unique tags
    all_tags = set()
    for col in tag_cols:
        df[col].apply(parse_tags).apply(all_tags.update)
    all_tags = sorted(all_tags)
    
    # Build binary matrix
    rows = []
    for _, row in df.iterrows():
        tag_set = set()
        for col in tag_cols:
            tag_set.update(parse_tags(row[col]))
        rows.append({tag: int(tag in tag_set) for tag in all_tags})
    
    result = pd.DataFrame(rows, index=df['content_id'])
    result.index.name = 'content_id'
    return result

df_tags = build_tag_matrix(df_sounds_music)

In [ ]:
# check if we can combine tags
# all_tags.sort()
print(all_tags)

df_features (binary, sparse)
    → PPMI (normalizes for popular tags)
        → TruncatedSVD (20 dims, dense)
            → X_dense (n_sounds × 20, L2-normalized)
                → content_sim score per candidate sound

```
score = α * content_sim      ← X_dense covers this
+ β * co_occ           ← from your listening sessions (sound pairs in same mix)
+ γ * pop_penalty       ← NOT computed yet
+ δ * diversity_penalty ← NOT computed yet
```

In [ ]:
# Compute PPMI for co-occurence matrix (note: use PPMI instead of TF-IDF because popular sounds are down-weighted)
# Other Alternative: Jaccard, Normalized PMI
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import numpy as np

X = df_tags.values.astype(float)   # (n_sounds, n_tags)
C = X @ X.T                            # (n_sounds, n_sounds)

# PPMI
total      = C.sum()
marginals  = C.sum(axis=1) / total
P_ij       = C / total
P_i_P_j    = np.outer(marginals, marginals)

with np.errstate(divide='ignore', invalid='ignore'):
    pmi = np.where(P_ij > 0, np.log2(P_ij / (P_i_P_j + 1e-10)), 0)

ppmi = np.clip(pmi, 0, None)   # Positive PMI: drop negative values

# SVD on PPMI matrix → dense embeddings
svd     = TruncatedSVD(n_components=20, random_state=42)
X_dense = normalize(svd.fit_transform(ppmi), norm='l2')

print(f"Explained variance: {svd.explained_variance_ratio_.sum():.1%}")


In [ ]:
total_sessions = len(df_listening)
sound_pop = df_listening_exploded['sounds'].value_counts() / total_sessions

# Log-normalize so mega-popular sounds don't dominate
pop_penalty = np.log1p(sound_pop)
pop_penalty = (pop_penalty - pop_penalty.mean()) / pop_penalty.std()  # z-score


In [ ]:
pop_penalty

In [ ]:
# diversity penalty
def redundancy(mix_sounds, candidate_id):
    """Max cosine similarity between candidate and any sound already in mix."""
    if candidate_id not in df_tags.index:
        return 0.0
    candidate_vec = X_dense[df_tags.index.get_loc(candidate_id)]
    scores = []
    for s in mix_sounds:
        if s in df_tags.index:
            mix_vec = X_dense[df_tags.index.get_loc(s)]
            scores.append(candidate_vec @ mix_vec)  # already L2-normalized
    return max(scores) if scores else 0.0


In [ ]:
# 1. Mix embedding = volume-weighted average of its sounds' embeddings
def mix_embedding(sounds_volumes: dict) -> np.ndarray:
    vecs, weights = [], []
    for sound_id, vol in sounds_volumes.items():
        if sound_id in df_tags.index:
            idx = df_tags.index.get_loc(sound_id)
            vecs.append(X_dense[idx])
            weights.append(vol)
    if not vecs:
        return None
    return normalize(np.average(vecs, axis=0, weights=weights).reshape(1, -1))[0]

# 2. content_sim(candidate) = cosine similarity to mix embedding
mix_vec = mix_embedding({"ambience.rain": 0.8, "ambience.ocean": 0.5})
content_sim = X_dense @ mix_vec   # (n_sounds,) — dot product = cosine sim since X_dense is L2-normalized


Matrix Sparsity

| Sparsity | Avg active tags (60 features) | Works for similarity?        |
|----------|-------------------------------|------------------------------|
| < 70%    | 18+                           | Rich signal                  |
| 70–85%   | 9–18                          | Good                         |
| 85–92%   | 5–9                           | Acceptable — your range      |
| 92–97%   | 2–4                           | Weak, many zero pairs        |
| > 97%    | < 2                           | Broken                       |

In [ ]:
# matrix sparsity: 
# P(at least 1 shared tag) ≈ 1 - (sparsity)^{avg tag per sound} ≈ 1 - (1 - 0.1)^6 ≈ 47% => Half sound pair have cosine similarity = 0 
sparsity = 1 - df_tags.values.mean()
print(f"Sparsity: {sparsity:.1%}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import numpy as np

X = normalize(df_tags.values.astype(float), norm='l2')
sim = cosine_similarity(X)

# Exclude self-similarity (diagonal)
np.fill_diagonal(sim, 0)

zero_pairs = (sim == 0).mean()
print(f"Fraction of sound pairs with zero similarity: {zero_pairs:.1%}")
# If this is > 40%, your content signal has big blind spots


In [ ]:
print(len(df_tags.columns)) 
print(len(set(all_tags))) # has nan

In [ ]:
df_tags_popularity = df_tags.sum(axis=0) / len(df_sounds_music)
df_tags_popularity

In [ ]:
# count number of tags per sounds
df_sounds_music_tag_count = df_tags.sum(axis=1)
print(f"min: {df_sounds_music_tag_count.min()}")
print(f"max: {df_sounds_music_tag_count.max()}")
print(f"avg: {df_sounds_music_tag_count.mean()}")
print(f"Sounds with <=3 tags: {(df_sounds_music_tag_count <= 3).sum()}")
print(f"Sounds with >=10 tags: {(df_sounds_music_tag_count >= 10).sum()}")


In [ ]:
# normalize tag counts: L2 normalization (unit vector) OR [TF-IDF] (better)
from sklearn.feature_extraction.text import TfidfTransformer

tfidf = TfidfTransformer(norm='l2', smooth_idf=True)
X_tfidf = tfidf.fit_transform(df_tags.values)  # sparse matrix

# Back to dense if needed
df_tags_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    index=df_tags.index,
    columns=df_tags.columns
)
df_tags_tfidf

In [ ]:
# GETTING DENSER: (opt 1) Using SVD to get a denser embedding
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=20, random_state=42) # note: 20-30 tags should be enough
X_dense = svd.fit_transform(X_tfidf)

In [ ]:
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.1%}") # Aim for > 60-70% explained variance
sim = cosine_similarity(X_dense)
np.fill_diagonal(sim, 0)
print(f"Sim Dense: {(sim == 0).mean()}") # Now every pair has a non-zero similarity

sparsity = 1 - X_dense.mean()
print(f"Sparsity: {sparsity:.1%}")

# svd.explained_variance_ratio_.cumsum()

In [ ]:
# GETTING DENSER: (opt 2) Rare threshold (skipped)

In [ ]:
# GETTING DENSER: (opt 3) Semantic Grouping (from chat) [TODO]
tag_groups = {
    'ADHD': 'Mental State',
    'Anxiety relief': 'Mental State',
    'Anger': 'Mental State',
    'Bored': 'Mental State',
    'Depressed': 'Mental State',
    'Grieving': 'Mental State',
    'Lack of Energy': 'Mental State',
    'Overthinking': 'Mental State',
    'Self-esteem': 'Mental State',

    'Healing': 'Wellness',
    'Pain Management': 'Wellness',
    'Tinnitus': 'Wellness',
    'Tinnitus Awareness Week': 'Wellness',
    "Women's health": 'Wellness',

    'Bedtime': 'Sleep Context',
    'Fall Asleep': 'Sleep Context',
    'Fall Back Asleep': 'Sleep Context',
    'Napping': 'Sleep Context',

    'NREM N1': 'Sleep Stage',
    'NREM N2': 'Sleep Stage',
    'NREM N3': 'Sleep Stage',
    'REM': 'Sleep Stage',

    'Calm': 'Mood',
    'De-Stress': 'Mood',
    'Relaxation': 'Mood',
    'Focus': 'Mood',
    'Grounding': 'Mood',
    'Positive': 'Mood',
    'Empowering': 'Mood',
    'Escapism': 'Mood',

    'ASMR': 'Audio Type',
    'Colored Noise': 'Audio Type',
    'Music': 'Audio Type',
    'Solfeggio': 'Audio Type',
    'Brainwaves': 'Audio Type',
    'Major Foley': 'Audio Type',
    'Minor Foley': 'Audio Type',
    'Multi-Voice': 'Audio Type',
    '3D': 'Audio Type',
    'Bilateral': 'Audio Type',

    'Indoor City': 'Environment',
    'Outdoor City': 'Environment',
    'Indoor Nature': 'Environment',
    'Outdoor Nature': 'Environment',
    'Underwater': 'Environment',
    'Outerspace': 'Environment',
    'North America': 'Environment',
    'No Location': 'Environment',

    'Morning': 'Time',
    'Daytime': 'Time',
    'Nighttime': 'Time',

    'Christmas': 'Seasonal/Cultural',
    'Halloween': 'Seasonal/Cultural',
    'Fall': 'Seasonal/Cultural',
    'Spring': 'Seasonal/Cultural',
    'Summer': 'Seasonal/Cultural',
    'Winter': 'Seasonal/Cultural',
    'Festive': 'Seasonal/Cultural',
    'Back to School': 'Seasonal/Cultural',
    'Exam Season': 'Seasonal/Cultural',
    'Pride Month': 'Seasonal/Cultural',
    "New Year's": 'Seasonal/Cultural',
    'Cultural': 'Seasonal/Cultural',

    'Magical': 'Aesthetic',
    'Retro': 'Aesthetic',
    'Dreamscape': 'Aesthetic',
    'Playful': 'Aesthetic',
    'Sentimental': 'Aesthetic',
}

In [ ]:
# import numpy as np
# from scipy.stats import zscore

# # sound_id → content_id mapping (exclude the 'unmapped' metadata key)
# # id_map = {k: v for k, v in sound_to_content_ids.items() if k != 'unmapped'}



# # Join: listening event rows → content metadata / tags
# # df_joined = df_listening_exploded.copy()
# # df_joined['content_id'] = df_joined['sounds'].map(id_map)

# # df_joined = df_joined.merge(
# #     df_sounds[['Content ID'] + TAG_COLS_EDA].rename(columns={'Content ID': 'content_id'}),
# #     on='content_id',
# #     how='left',
# # )

# # Sound-level metadata: one row per unique sound_id
# sound_play_count = df_listening_exploded['sounds'].value_counts()
# df_sound_meta = (
#     df_joined[['sounds', 'content_id'] + TAG_COLS_EDA]
#     .drop_duplicates(subset='sounds')
#     .copy()
# )
# df_sound_meta['play_count'] = df_sound_meta['sounds'].map(sound_play_count).fillna(0)
# df_sound_meta['play_rate'] = df_sound_meta['play_count'] / sound_play_count.sum()

# n_matched = df_joined['ℹ️ Themes'].notna().sum()
# print(f"Session rows with matched tags: {n_matched:,} / {len(df_joined):,} ({n_matched/len(df_joined):.1%})")
# df_joined.head()

### Linking listening id with content ID

In [ ]:
print(len(df_listening_exploded['sounds'].unique()))
print(len(df_sounds_music))
print(len(sounds_ids))

### EDA

- Should we try to unmapped sounds_id? YES

In [ ]:
unmapped_sound_ids = sound_to_content_ids['unmapped']
unmapped_mask = df_listening_exploded['sounds'].isin(unmapped_sound_ids)
df_unmapped_sounds_exploded = df_listening_exploded[unmapped_mask]

In [ ]:
df_unmapped_sounds_exploded

In [ ]:
# --- Tag Coverage: how many unique sounds have each tag column filled ---
n_sounds = len(df_sound_meta)
print(f"=== Tag Coverage ({n_sounds} unique sounds) ===")
for col in TAG_COLS_EDA:
    n_covered = df_sound_meta[col].notna().sum()
    print(f"  {col:<30} {n_covered:3d} / {n_sounds} ({n_covered/n_sounds:.0%})")

# --- Tag frequency distribution (top 10 per column) ---
print("\n=== Tag Value Counts (top 10 per column) ===")
for col in TAG_COLS_EDA:
    counts = (
        df_sound_meta[col]
        .dropna()
        .str.split(',')
        .explode()
        .str.strip()
        .value_counts()
        .head(10)
    )
    print(f"\n{col}:\n{counts.to_string()}")

##### TAG x Listening

In [ ]:
# --- Listening Rate per Tag (z-score normalized within each tag type) ---
#
# play_rate(sound) = play_count(sound) / total_sound_plays   [0..1]
# mean_play_rate(tag) = mean over all sounds with that tag
# normalized  = z-score within tag_type → removes scale differences across columns

frames = []
for col, label in TAG_COL_LABELS.items():
    df_t = (
        df_sound_meta[['sounds', 'play_count', 'play_rate', col]]
        .dropna(subset=[col])
        .assign(tag=lambda x: x[col].str.split(','))
        .explode('tag')
        .assign(tag=lambda x: x['tag'].str.strip(), tag_type=label)
    )
    frames.append(df_t[['sounds', 'play_count', 'play_rate', 'tag', 'tag_type']])

df_tags_long = pd.concat(frames, ignore_index=True)

tag_stats = (
    df_tags_long.groupby(['tag_type', 'tag'])
    .agg(
        n_sounds     = ('sounds',     'nunique'),
        total_plays  = ('play_count', 'sum'),
        mean_play_rate = ('play_rate', 'mean'),
    )
    .reset_index()
)

# Z-score within each tag_type (guards against single-value groups)
tag_stats['normalized_play_rate'] = (
    tag_stats.groupby('tag_type')['mean_play_rate']
    .transform(lambda x: zscore(x, ddof=1) if x.nunique() > 1 else pd.Series(0.0, index=x.index))
)

tag_stats.sort_values('normalized_play_rate', ascending=False)

In [ ]:
# --- Mean Volume per Tag (z-score normalized within each tag type) ---
#
# High normalized volume → users place these sounds "front and centre" (foreground)
# Low normalized volume  → background / ambient role
# Useful feature: tells the model whether a recommended sound should be dominant or subtle

vol_frames = []
for col, label in TAG_COL_LABELS.items():
    df_t = (
        df_joined[['sounds', 'sounds_volume', col]]
        .dropna(subset=[col])
        .assign(tag=lambda x: x[col].str.split(','))
        .explode('tag')
        .assign(tag=lambda x: x['tag'].str.strip(), tag_type=label)
    )
    vol_frames.append(df_t[['sounds', 'sounds_volume', 'tag', 'tag_type']])

df_vol_long = pd.concat(vol_frames, ignore_index=True)

vol_stats = (
    df_vol_long.groupby(['tag_type', 'tag'])
    .agg(
        mean_volume    = ('sounds_volume', 'mean'),
        std_volume     = ('sounds_volume', 'std'),
        n_observations = ('sounds_volume', 'count'),
    )
    .reset_index()
)

vol_stats['normalized_volume'] = (
    vol_stats.groupby('tag_type')['mean_volume']
    .transform(lambda x: zscore(x, ddof=1) if x.nunique() > 1 else pd.Series(0.0, index=x.index))
)

vol_stats.sort_values('normalized_volume', ascending=False)

In [ ]:
# --- Tag Co-occurrence in Mixes + PMI (Pointwise Mutual Information) ---
#
# Raw count tells you which tags appear together most often.
# PMI normalises by individual tag frequency → surfaces *unexpectedly* tight pairs,
# not just pairs that co-occur a lot because both tags are globally common.
#
# PMI(a,b) = log2( P(a,b) / (P(a) * P(b)) )
#   > 0  → tags co-occur more than chance
#   = 0  → independent
#   < 0  → tags avoid each other

from itertools import combinations
from collections import Counter

def get_session_tags(group):
    tags = set()
    for col in TAG_COLS_EDA:
        for val in group[col].dropna():
            tags.update(t.strip() for t in val.split(','))
    return tags

# Group by original session index (preserved through the merge)
session_tags = df_joined.groupby(df_joined.index).apply(get_session_tags)

# Raw co-occurrence counts
co_occ = Counter(
    pair
    for tags in session_tags
    for pair in combinations(sorted(tags), 2)
    if len(tags) >= 2
)

# Marginal tag frequencies
tag_freq = Counter(tag for tags in session_tags for tag in tags)
n_sessions = len(session_tags)

def pmi(tag_a, tag_b, co_count):
    p_ab = co_count / n_sessions
    p_a  = tag_freq[tag_a] / n_sessions
    p_b  = tag_freq[tag_b] / n_sessions
    return np.log2(p_ab / (p_a * p_b + 1e-10))

df_cooc = pd.DataFrame(
    [(a, b, c) for (a, b), c in co_occ.most_common(200)],
    columns=['tag_a', 'tag_b', 'co_count'],
)
df_cooc['pmi'] = df_cooc.apply(
    lambda r: pmi(r['tag_a'], r['tag_b'], r['co_count']), axis=1
)
# Normalise raw count (z-score) for easy comparison
df_cooc['normalized_count'] = zscore(df_cooc['co_count'], ddof=1)

print(f"Unique tag pairs observed: {len(df_cooc):,}")
df_cooc.sort_values('pmi', ascending=False).head(20)

In [ ]:
import matplotlib.pyplot as plt

# Listening rate (left) vs. volume (right) — one subplot per tag type
fig, axes = plt.subplots(len(TAG_COL_LABELS), 2, figsize=(14, 4 * len(TAG_COL_LABELS)))

for row, (col, label) in enumerate(TAG_COL_LABELS.items()):

    # --- listening rate ---
    ax = axes[row, 0]
    sub = (
        tag_stats[tag_stats['tag_type'] == label]
        .sort_values('normalized_play_rate')
        .tail(15)
    )
    colors = ['#d73027' if v > 0 else '#4575b4' for v in sub['normalized_play_rate']]
    ax.barh(sub['tag'], sub['normalized_play_rate'], color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'[{label}] Listening Rate (z-score)')
    ax.set_xlabel('Normalized play rate')

    # --- mean volume ---
    ax = axes[row, 1]
    sub_v = (
        vol_stats[vol_stats['tag_type'] == label]
        .sort_values('normalized_volume')
        .tail(15)
    )
    colors_v = ['#d73027' if v > 0 else '#4575b4' for v in sub_v['normalized_volume']]
    ax.barh(sub_v['tag'], sub_v['normalized_volume'], color=colors_v)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'[{label}] Mean Volume (z-score)')
    ax.set_xlabel('Normalized volume')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Content-based similarity between sounds using tag vectors
sim_matrix = cosine_similarity(df_features.values)
df_sim = pd.DataFrame(sim_matrix, index=df_features.index, columns=df_features.index)

def top_similar_sounds(sound_id, k=5):
    if sound_id not in df_sim.index:
        return f"'{sound_id}' not in feature matrix"
    return df_sim[sound_id].drop(sound_id).sort_values(ascending=False).head(k)

# Example: most tag-similar sounds to ambience.rain
top_similar_sounds('ambience.rain')

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

def build_tag_vectors(df_meta, col, label):
    tag_lists = (
        df_meta[col]
        .fillna('')
        .str.split(',')
        .apply(lambda x: [t.strip() for t in x if t.strip()])
    )
    mlb = MultiLabelBinarizer()
    vectors = mlb.fit_transform(tag_lists)
    cols = [f"{label}__{c}" for c in mlb.classes_]
    return pd.DataFrame(vectors, index=df_meta['sounds'].values, columns=cols)

feature_frames = [
    build_tag_vectors(df_sound_meta.reset_index(drop=True), col, label)
    for col, label in TAG_COL_LABELS.items()
]
df_features = pd.concat(feature_frames, axis=1).fillna(0).astype(int)
df_features.index.name = 'sound_id'

print(f"Feature matrix: {df_features.shape[0]} sounds × {df_features.shape[1]} binary tag features")
print(f"Sparsity: {(df_features == 0).sum().sum() / df_features.size:.1%} zeros")
df_features.head()

### Sound Tag Feature Matrix

Binary tag vectors per sound → used as item embeddings for content-based similarity.
Each sound becomes a sparse binary vector: `1` if the sound has that tag, `0` otherwise.

```
def score(mix_sounds_volumes, candidate_ids):
    interaction_vec = build_interaction_vector(mix_sounds_volumes)
    mix_emb = build_mix_embedding(mix_sounds_volumes)

    als       = normalize(item_factors @ (foldin_matrix @ interaction_vec))
    content   = X_dense @ mix_emb          # (n_sounds,)
    pop       = pop_penalty.reindex(candidate_ids).fillna(0).values
    diversity = -(X_dense @ mix_emb)       # negative: penalize redundancy

    return 0.5 * als + 0.3 * content + 0.1 * pop + 0.1 * diversity

```